# Artefacts in ODYSSEY (H2)

H2 (#19). The methods come from the literature collected in `docs/method.md`, *Cleaning of ODYSSEY (H2)*;
the review of #74 chose EURADCLIM (Overeem et al. 2023): the Gabella filter on each composite, a satellite
mask, `histo_cut` on annual sums, and Wagner-style frequency maps for the catalogue.

1. **Gabella at 2 km** (review of #74, point 5). The filter was set on single-radar 1 km maps (Gabella and
   Notarpietro 2002); at 2 km its 5 × 5 window covers 10 km and the smallest kept echo is about 44 km².
   How much convective rain does it remove?

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
from collections import defaultdict

import numpy as np
from wradlib.classify import filter_gabella

from opera_ds.classify import obs_classes
from opera_ds.odim_io import index_files
from opera_ds.validate import read_degraded

## 1. Gabella at 2 km on convective days

- **Data:** CIRRUS degraded to 2 km by `zmean`, the H1 stand-in for ODYSSEY (`docs/method.md`, *Degraded
  CIRRUS as a stand-in for ODYSSEY*), at :00, :15, :30 and :45, over the Belgian crop.
- **Days:** the 5 days of June–July 2025 with most Steiner convective pixels in the H1 run (rows at
  `alpha` = 0, which are the same for every method).
- **Filter:** `wradlib.classify.filter_gabella` with its defaults (5 × 5 window, `tr1` = 6 dB, `n_p` = 6,
  `tr2` = 1.3), `cartesian=True`, as in EURADCLIM. Field in dBZ, `undetect` and values below 0 dBZ set to
  0 dBZ (`thrsnorain` = 0), NaN outside coverage.
- **Classes:** Steiner on the 2 km field (`opera_ds.classify`, as in H1).
- **Rain:** Marshall–Palmer with dBZ capped at 55 (Imhoff et al. 2020), on echoes ≥ 10 dBZ, the wet
  threshold of `run_validation`.

Share of rain (sum of R) and of pixels flagged by the filter, per class.

In [ ]:
WET_DBZ = 10.0
N_DAYS = 5


def rain(dbz):
    return (10 ** (np.minimum(dbz, 55.0) / 10) / 200.0) ** (1 / 1.6)


days = defaultdict(int)
for mon in ("20250601-20250701", "20250701-20250801"):
    with open(RESULTS_DIR / "h1_degraded_cirrus" / mon / "per_time.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            if r["method"] == "nearest|convective" and float(r["alpha"]) == 0.0:
                days[r["time"][:10]] += int(r["n_wet"])
top = sorted(days, key=lambda d: -days[d])[:N_DAYS]
print("days (convective pixels, H1, alpha = 0):", [(d, days[d]) for d in top])

files = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
todo = [(f"{t:%Y-%m-%d}", str(p)) for t, p in sorted(files.items()) if t.minute % 15 == 0 and f"{t:%Y-%m-%d}" in top]
print(len(todo), "frames")

In [ ]:
tot = defaultdict(lambda: defaultdict(lambda: np.zeros(4)))   # rain, rain flagged, pixels, pixels flagged
for d, path in todo:
    f = read_degraded(path, factor=2).to_field(0.0)
    cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
    flag = filter_gabella(f.copy(), wsize=5, cartesian=True)
    wet = np.isfinite(f) & (f >= WET_DBZ)
    r = np.where(wet, rain(np.nan_to_num(f)), 0.0)
    for name, m in (("all", wet), ("convective", wet & cls["convective"]), ("stratiform", wet & cls["stratiform"])):
        v = np.array([r[m].sum(), r[m & flag].sum(), m.sum(), (m & flag).sum()], float)
        tot[d][name] += v
        tot["all days"][name] += v

print(f"{'day':12s} {'class':11s} {'rain removed':>13s} {'pixels removed':>15s} {'wet pixels':>11s}")
for d in top + ["all days"]:
    for k in ("all", "convective", "stratiform"):
        s, sr, n, nr = tot[d][k]
        print(f"{d:12s} {k:11s} {sr / s:12.1%} {nr / n:14.1%} {int(n):11d}")

### What is seen

Run on 9/11/2026. Over the five days the filter removes 4.9% of the convective rain (2.5% of the
convective pixels) and 0.9% of the stratiform rain; between 4.0% and 6.3% of the convective rain per day.
These are rain echoes of CIRRUS, so whatever the filter removes here is real rain lost, not clutter.